In [ ]:
#Seetting up a safe and standarized library to keep consistency

In [1]:
   import sys
   print(sys.executable)

C:\Users\zyany\GitHub\CENACE-CHALLENGE\arkham-env\Scripts\python.exe


In [ ]:
#All looking good

In [ ]:
#Setting up API to get updated information from site

In [3]:
#Importing necessary lybraries
import json, time
from datetime import date, timedelta
from pathlib import Path

#Seting up the data that will be retrieved from CENACE's live database, parameters are set up on yaml file.
import requests
import yaml
from tenacity import retry, stop_after_attempt, wait_exponential
from tqdm import tqdm

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

api   = cfg["api"]
nodes = cfg["nodes"]
start = date.fromisoformat(cfg["period"]["start"])
end   = date.fromisoformat(cfg["period"]["end"])

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(len(nodes), "nodes |", start, "→", end)

9 nodes | 2024-01-01 → 2025-06-30


In [ ]:
#Seting up the function to iterate through the api url requests 

In [4]:
def build_url(nodes, d_start, d_end):
    nodos = ",".join(nodes)          # comma-separated, no spaces (manual's rule)
    return (f"{api['base_url']}/{api['sistema']}/{api['proceso']}/{nodos}/"
            f"{d_start:%Y/%m/%d}/{d_end:%Y/%m/%d}/{api['formato']}")

print(build_url(nodes, start, start))

https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/2024/01/01/2024/01/01/JSON


In [ ]:
#Making a test call to go through the data and make sure functions are properly working

In [5]:
r = requests.get(build_url(nodes, start, start), timeout=120)
print("Status:", r.status_code)
test = r.json()
print(test.keys() if isinstance(test, dict) else type(test))

Status: 200
dict_keys(['nombre', 'proceso', 'sistema', 'area', 'Resultados', 'status'])


In [6]:
type(test)

dict

In [ ]:
#information structure is a dictionay so now will call the keys to see what's inside the dictionary

In [7]:
test.keys()

dict_keys(['nombre', 'proceso', 'sistema', 'area', 'Resultados', 'status'])

In [ ]:
#The data we are actually interested in would be under the "Resultados" key, so will explore that one

In [8]:
r = test["Resultados"]
type(r), len(r)

(list, 9)

In [ ]:
#It is a list of 9 which match the 9 nodes, we will take a look at the 1st row

In [9]:
r[0].keys()

dict_keys(['clv_nodo', 'Valores'])

In [ ]:
#It's giving another dictionary, so, I will look at the structure to get a better glance of indedntation and nesting

In [10]:
import json
print(json.dumps(test, indent=2, ensure_ascii=False)[:2000])

{
  "nombre": "PML",
  "proceso": "MDA",
  "sistema": "SIN",
  "area": "PÚBLICA",
  "Resultados": [
    {
      "clv_nodo": "06AEO-115",
      "Valores": [
        {
          "fecha": "2024-01-01",
          "hora": "1",
          "pml": "279.92",
          "pml_ene": "370.71",
          "pml_per": "-43.69",
          "pml_cng": "-47.1"
        },
        {
          "fecha": "2024-01-01",
          "hora": "2",
          "pml": "279.21",
          "pml_ene": "311.75",
          "pml_per": "-32.54",
          "pml_cng": "0"
        },
        {
          "fecha": "2024-01-01",
          "hora": "3",
          "pml": "274.23",
          "pml_ene": "305.52",
          "pml_per": "-31.28",
          "pml_cng": "0"
        },
        {
          "fecha": "2024-01-01",
          "hora": "4",
          "pml": "264.01",
          "pml_ene": "294.23",
          "pml_per": "-30.22",
          "pml_cng": "0"
        },
        {
          "fecha": "2024-01-01",
          "hora": "5",
          

In [ ]:
#API worked fine :) important to note that every value is text which will be an issue if left like this when working with the data